# Module 7 — Day 3: Multi-step Retrieval & Self-Correction

**Goal:** Build an agent that retrieves evidence, checks whether the evidence is sufficient, reformulates the query when it is not, retrieves again, and stops safely.

**Today’s workflow:** Retrieve → Assess evidence → Reformulate → Re-retrieve → Verify → Respond.

**Prerequisites:** Day 1 and Day 2. Use the same Module 6 invoice corpus. This notebook uses local TF-IDF retrieval and transparent rules, so it does not require a GPU or API key. The sufficiency checker is a practical heuristic, not a language-model judge.

## 1. Install dependencies
Install the local PDF and retrieval packages used in this exercise.

In [1]:
!pip -q install pymupdf pandas scikit-learn

## 2. Imports and workspace
Set up the document records and an execution trace.

In [2]:
import json, re
from pathlib import Path
from datetime import datetime
import fitz, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

WORK = Path("module7_day3")
WORK.mkdir(exist_ok=True)
RECORDS, AGENT_TRACE = [], []

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


## 3. Upload the document corpus
Upload `sample_multimodal_invoice.pdf`. You may also upload `day4_retrieval_results.json` from Module 6; when present, its text records are used instead of extracting PDF text.

In [16]:
import fitz
from pathlib import Path
import pandas as pd

def load_records():
    pdf_file = Path("sample_multimodal_invoice.pdf")
    rows = []

    with fitz.open(pdf_file) as doc:
        for page_num, page in enumerate(doc, start=1):
            page_text = page.get_text("text").strip()

            if page_text:
                rows.append({
                    "source_id": f"PDF-P{page_num}",
                    "source_doc": pdf_file.name,
                    "page": page_num,
                    "text": page_text
                })

    RECORDS.clear()
    RECORDS.extend(rows)

    return {
        "record_count": len(rows),
        "pages": [r["page"] for r in rows]
    }

print(load_records())
display(pd.DataFrame([
    {k: r.get(k) for k in ["source_id", "source_doc", "page"]}
    for r in RECORDS
]))

{'record_count': 2, 'pages': [1, 2]}


,source_id,source_doc,page
0,PDF-P1,sample_multimodal_invoice.pdf,1
1,PDF-P2,sample_multimodal_invoice.pdf,2


In [17]:
print("PDF path:", pdf_path)
print("File exists:", Path(pdf_path).exists())

with fitz.open(pdf_path) as doc:
    print("Pages:", len(doc))
    print("First page text:", repr(doc[0].get_text()[:500]))

PDF path: sample_multimodal_invoice.pdf
File exists: True
Pages: 2
First page text: 'Synthetic sample document — for notebook testing\nPage 1\nTAX INVOICE\nOriginal for recipient\nSupplier\nMeridian Research Labs Pvt. Ltd.\nPlot 18, Sector 44\nGurugram, Haryana 122003\nGSTIN: 06AABCM1234F1Z5\nEmail: billing@meridianlabs.example\nBill To\nNorthstar Office Supplies\nUnit 402, Tech Park\nJaipur, Rajasthan 302017\nGSTIN: 08AACCN5678L1Z2\nAttn: Accounts Payable\nInvoice No.\nMRL-2026-1184\nInvoice Date\n18 Nov 2026\nPO Number\nNS-PO-7741\nDue Date\n03 Dec 2026\nPlace of Supply\nRajasthan\nCurrency\nINR (I)\nIte'


## 4. Create the first-pass retriever
TF-IDF provides a small, inspectable baseline. Each hit retains a source ID and page so later checks can verify provenance.

In [18]:
def retrieve(query, top_k=4, page=None):
    corpus = [r for r in RECORDS if page is None or r.get("page") == page]
    if not corpus:
        return {"ok":False,"query":query,"hits":[],"reason":"No records matched the page filter."}
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    matrix = vectorizer.fit_transform([r["text"] for r in corpus] + [query])
    scores = cosine_similarity(matrix[-1], matrix[:-1]).ravel()
    order = scores.argsort()[::-1][:top_k]
    hits = [{
        "source_id":corpus[i]["source_id"],"source_doc":corpus[i]["source_doc"],
        "page":corpus[i]["page"],"score":round(float(scores[i]),4),
        "text":corpus[i]["text"][:1400]
    } for i in order]
    return {"ok":True,"query":query,"hits":hits}
print("Retriever ready. Records:",len(RECORDS))

Retriever ready. Records: 2


## 5. Define an evidence sufficiency check
The checker looks for useful similarity scores, non-empty text, and source metadata. It also checks whether the query’s important terms appear in the retrieved evidence. These rules are intentionally visible and adjustable.

In [19]:
STOP_WORDS = {"what","which","where","when","who","how","does","show","tell","find","compare",
              "the","and","for","with","from","this","that","about","into","page"}

def query_terms(query):
    return {w for w in re.findall(r"[a-z0-9]+", query.lower()) if len(w) > 2 and w not in STOP_WORDS}

def assess_evidence(query, retrieval, min_score=0.03, min_hits=1):
    hits = retrieval.get("hits", [])
    valid = [h for h in hits if h.get("text","").strip() and h.get("source_id") and h.get("score",0) >= min_score]
    terms = query_terms(query)
    combined = " ".join(h["text"].lower() for h in valid)
    covered = sorted(t for t in terms if t in combined)
    missing = sorted(terms - set(covered))
    sufficient = len(valid) >= min_hits and len(covered) >= max(1, min(2, len(terms)))
    return {
        "sufficient": sufficient, "valid_hit_count": len(valid),
        "covered_terms": covered, "missing_terms": missing,
        "reason": "Evidence meets heuristic threshold." if sufficient else "Evidence is weak or misses key query terms."
    }

test_query = "What is the invoice total and who is the supplier?"
first = retrieve(test_query)
print(json.dumps(assess_evidence(test_query, first), indent=2))

{
  "sufficient": true,
  "valid_hit_count": 1,
  "covered_terms": [
    "invoice",
    "supplier",
    "total"
  ],
  "missing_terms": [],
  "reason": "Evidence meets heuristic threshold."
}


## 6. Reformulate a weak query
A correction step should change the search wording, not merely repeat the same query. This simple reformulator adds likely synonyms and extracts key nouns. Later, an LLM planner can generate richer rewrites.

In [20]:
SYNONYMS = {
    "total": ["grand total","amount due","invoice amount","sum"],
    "supplier": ["vendor","seller","from"],
    "chart": ["graph","figure","plot"],
    "table": ["line items","tabular data","breakdown"],
    "date": ["invoice date","billing date"]
}

def reformulate_query(query, assessment):
    terms = query_terms(query)
    expansions = []
    for term in terms:
        expansions.extend(SYNONYMS.get(term, []))
    missing = assessment.get("missing_terms", [])
    focused = " ".join(missing) if missing else query
    rewrite = f"{query} {' '.join(expansions)} {focused}".strip()
    rewrite = re.sub(r"\\s+", " ", rewrite)
    return rewrite

weak_assessment = {"missing_terms":["supplier","total"]}
print(reformulate_query(test_query, weak_assessment))

What is the invoice total and who is the supplier? vendor seller from grand total amount due invoice amount sum supplier total


## 7. Build the iterative retrieval controller
The controller retries only when evidence is insufficient, caps the number of attempts, and records each query, result, assessment, and rewrite.

In [21]:
def iterative_retrieve(query, max_attempts=3, top_k=5, min_score=0.03):
    attempts = []
    current_query = query
    seen_queries = set()

    for attempt_no in range(1, max_attempts + 1):
        if current_query.lower() in seen_queries:
            break
        seen_queries.add(current_query.lower())

        result = retrieve(current_query, top_k=top_k)
        assessment = assess_evidence(query, result, min_score=min_score)
        attempt = {
            "attempt":attempt_no, "query":current_query,
            "retrieval":result, "assessment":assessment
        }
        attempts.append(attempt)

        if assessment["sufficient"]:
            break
        if attempt_no == max_attempts:
            break

        current_query = reformulate_query(current_query, assessment)

    final_assessment = attempts[-1]["assessment"] if attempts else {"sufficient":False}
    output = {
        "original_query":query, "attempts":attempts,
        "sufficient":final_assessment.get("sufficient",False),
        "attempt_count":len(attempts),
        "stop_reason":"sufficient evidence" if final_assessment.get("sufficient") else "attempt limit or repeated query"
    }
    AGENT_TRACE.append(output)
    return output

result = iterative_retrieve("What is the invoice total and who is the supplier?")
print("Attempts:",result["attempt_count"],"| Sufficient:",result["sufficient"],"| Stop:",result["stop_reason"])

Attempts: 1 | Sufficient: True | Stop: sufficient evidence


## 8. Inspect every retrieval attempt
Review how the query changed and whether each attempt improved evidence coverage.

In [22]:
attempt_rows = []
for a in result["attempts"]:
    attempt_rows.append({
        "attempt":a["attempt"],"query":a["query"],
        "sufficient":a["assessment"]["sufficient"],
        "hits":a["assessment"]["valid_hit_count"],
        "covered":", ".join(a["assessment"]["covered_terms"]),
        "missing":", ".join(a["assessment"]["missing_terms"])
    })
display(pd.DataFrame(attempt_rows))

,attempt,query,sufficient,hits,covered,missing
0,1,What is the invoice total and who is the suppl...,True,1,"invoice, supplier, total",


## 9. Verify source provenance and answerability
Before drafting an answer, verify that each accepted hit has a source ID, document name, page, and non-empty content. Reject malformed evidence.

In [23]:
def verify_sources(attempt_result):
    if not attempt_result["attempts"]:
        return {"valid":False,"verified_hits":[],"errors":["No retrieval attempts ran."]}
    final_hits = attempt_result["attempts"][-1]["retrieval"].get("hits", [])
    verified, errors = [], []
    for hit in final_hits:
        required = ["source_id","source_doc","page","text"]
        missing = [k for k in required if hit.get(k) is None or hit.get(k) == ""]
        if missing:
            errors.append({"source_id":hit.get("source_id"),"missing_fields":missing})
        else:
            verified.append(hit)
    return {"valid":len(verified)>0,"verified_hits":verified,"errors":errors}

verification = verify_sources(result)
print("Sources verified:",verification["valid"])
print("Verified source IDs:",[h["source_id"] for h in verification["verified_hits"]])

Sources verified: True
Verified source IDs: ['PDF-P1', 'PDF-P2']


## 10. Generate a grounded evidence summary
This function summarizes retrieved snippets with citations. It deliberately avoids inventing facts: if the evidence is insufficient, it says so.

In [24]:
def grounded_response(query, attempt_result, verification):
    if not attempt_result["sufficient"] or not verification["valid"]:
        return "I could not find sufficient verified evidence to answer confidently. Try a more specific query or provide the relevant document."
    lines = ["Retrieved evidence for: " + query]
    for hit in verification["verified_hits"][:4]:
        lines.append(f"- {hit['text'][:550]} [Source: {hit['source_id']}, {hit['source_doc']}, page {hit['page']}]")
    lines.append("Use the cited excerpts as evidence; verify exact numeric values against the source document.")
    return "\\n".join(lines)

answer = grounded_response(test_query,result,verification)
print(answer)

Retrieved evidence for: What is the invoice total and who is the supplier?\n- Synthetic sample document — for notebook testing
Page 1
TAX INVOICE
Original for recipient
Supplier
Meridian Research Labs Pvt. Ltd.
Plot 18, Sector 44
Gurugram, Haryana 122003
GSTIN: 06AABCM1234F1Z5
Email: billing@meridianlabs.example
Bill To
Northstar Office Supplies
Unit 402, Tech Park
Jaipur, Rajasthan 302017
GSTIN: 08AACCN5678L1Z2
Attn: Accounts Payable
Invoice No.
MRL-2026-1184
Invoice Date
18 Nov 2026
PO Number
NS-PO-7741
Due Date
03 Dec 2026
Place of Supply
Rajasthan
Currency
INR (I)
Items and Services
#
Description
HSN/SAC
Qty
Unit Pri [Source: PDF-P1, sample_multimodal_invoice.pdf, page 1]\n- Synthetic sample document — for notebook testing
Page 2
Service and Delivery Details
Supporting schedule for invoice MRL-2026-1184
Delivery Schedule
Delivery Ref.
Item
Delivery Date
Location
Status
DLV-8812
Workstations (2 units)
17 Nov 2026
Jaipur office
Delivered
DLV-8813
Monitors (3 units)
17 Nov 2026
Jaipur

## 11. Demonstrate a deliberate first-pass failure
Constrain retrieval to a page that does not exist. The controller should detect the empty result, reformulate, and retry without a page restriction. This demonstrates recovery from an overly narrow first attempt.

In [25]:
def retrieve_with_page_fallback(query, requested_page, max_attempts=2):
    attempts=[]
    first=retrieve(query,page=requested_page)
    assessment=assess_evidence(query,first)
    attempts.append({"attempt":1,"page_filter":requested_page,"retrieval":first,"assessment":assessment})
    if assessment["sufficient"]:
        return {"attempts":attempts,"recovered":False}
    second_query=reformulate_query(query,assessment)
    second=retrieve(second_query,page=None)
    second_assessment=assess_evidence(query,second)
    attempts.append({"attempt":2,"page_filter":None,"query":second_query,
                     "retrieval":second,"assessment":second_assessment})
    return {"attempts":attempts,"recovered":second_assessment["sufficient"]}

recovery_demo=retrieve_with_page_fallback("invoice total supplier amount",requested_page=999)
print("Recovered:",recovery_demo["recovered"])
for a in recovery_demo["attempts"]:
    print("Attempt",a["attempt"],"| page filter:",a.get("page_filter"),
          "| sufficient:",a["assessment"]["sufficient"])

Recovered: True
Attempt 1 | page filter: 999 | sufficient: False
Attempt 2 | page filter: None | sufficient: True


## 12. Add a hard stop and inspect loop safety
A bounded loop must stop when evidence is sufficient, the attempt budget is exhausted, or a query repeats. This protects against runaway tool use.

In [26]:
def loop_safety_report(run):
    queries=[a["query"].lower() for a in run["attempts"]]
    return {
        "attempts":len(queries),
        "unique_queries":len(set(queries)),
        "no_repeated_queries":len(queries)==len(set(queries)),
        "within_limit":len(queries)<=3,
        "stopped":run["stop_reason"]
    }
print(loop_safety_report(result))

{'attempts': 1, 'unique_queries': 1, 'no_repeated_queries': True, 'within_limit': True, 'stopped': 'sufficient evidence'}


## 13. Evaluate several query types
Run a small set of questions and compare attempt counts and sufficiency. These are diagnostic examples, not a statistically meaningful benchmark.

In [27]:
evaluation_queries = [
    "What is the invoice total?",
    "Who is the supplier?",
    "What is the delivery address?",
    "Compare the chart with the totals table."
]
evaluation=[]
for q in evaluation_queries:
    r=iterative_retrieve(q,max_attempts=3)
    evaluation.append({"query":q,"attempts":r["attempt_count"],"sufficient":r["sufficient"],"stop_reason":r["stop_reason"]})
display(pd.DataFrame(evaluation))

,query,attempts,sufficient,stop_reason
0,What is the invoice total?,1,True,sufficient evidence
1,Who is the supplier?,3,False,attempt limit or repeated query
2,What is the delivery address?,1,True,sufficient evidence
3,Compare the chart with the totals table.,3,False,attempt limit or repeated query


## 14. Save the trace and results
Persist retrieval attempts, evidence assessments, and the recovery demonstration for later evaluation.

In [28]:
artifact={
    "main_run":result,
    "source_verification":{"valid":verification["valid"],"verified_source_ids":[h["source_id"] for h in verification["verified_hits"]],"errors":verification["errors"]},
    "grounded_response":answer,
    "recovery_demo":recovery_demo,
    "evaluation":evaluation
}
output_path=WORK/"day3_iterative_retrieval_trace.json"
output_path.write_text(json.dumps(artifact,indent=2,ensure_ascii=False),encoding="utf-8")
print("Saved:",output_path)

Saved: module7_day3/day3_iterative_retrieval_trace.json
